# btpvf v0 resynthesis

**Generated by `btpvf kaggle run` (`experiments.kaggle_harness`). Do not edit here** -- edit the harness and regenerate, or the repo stops holding what runs.

The code comes from the private `btpvf-code` Dataset built from the working tree or from `git archive` of a commit, **not** from GitHub: a run launched after an edit would otherwise execute the last pushed commit. The snapshot's mode, sha and content hash are printed below and recorded in the manifest's provenance sidecar.

**INV-13.** `griffin_lim`, `bigvgan_112m` and `vocos` share one environment here. That is the one stated exception, because BigVGAN and Vocos were *reported* by the team to coexist in testing; the repository has not measured it. Preflight refuses any other combination.

**Accelerator.** Pushed with `enable_gpu = true`.

In [ ]:
# --- GPU guard (generated; see experiments.kaggle_harness) ---
import torch

EXPECT_GPU = True
MIN_CAPABILITY = 7.0

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if EXPECT_GPU:
    if not torch.cuda.is_available():
        raise SystemExit("pushed with enable_gpu=true but no GPU is visible")
    major, minor = torch.cuda.get_device_capability(0)
    capability = major + minor / 10.0
    print("device:", torch.cuda.get_device_name(0), "| compute capability:", capability)
    if capability < MIN_CAPABILITY:
        raise SystemExit(
            f"Refusing to run on compute capability {capability} (< {MIN_CAPABILITY}). "
            "Kaggle assigned a P100; recent PyTorch builds ship no kernels for it, so "
            "this would fail deep inside the first convolution instead of here. "
            "Restart the session to be reassigned, usually a T4."
        )
else:
    print("CPU kernel: no accelerator requested, so this run costs no GPU quota.")
    if torch.cuda.is_available():
        print("note: a GPU is attached but unused; push without --gpu to free the quota.")


In [ ]:
# --- install the code snapshot (generated) ---
import json, shutil, subprocess, sys, zipfile
from pathlib import Path

# Nothing about where Kaggle puts things is assumed here, because every assumption
# has already cost a run. Two of them:
#   * an attached dataset mounts at /kaggle/input/datasets/<owner>/<slug>/ in this
#     account's image, not the /kaggle/input/<slug>/ the docs imply;
#   * Kaggle EXPANDS an uploaded .zip into a directory named after it with the
#     extension stripped, so btpvf_code.zip arrives as btpvf_code/.
# So: find the mount by the provenance file it always carries, then find the code
# by the pyproject.toml it must contain. Bounded globs, never rglob, which would
# walk every wav of an attached corpus.
_root = Path("/kaggle/input")
_found = [h for d in (1, 2, 3, 4)
          for h in sorted(_root.glob("/".join(["*"] * d) + "/btpvf_snapshot.json"))]
if not _found:
    raise SystemExit("no btpvf_snapshot.json under %s; attached: %s" % (
        _root, sorted(p.name for p in _root.iterdir()) if _root.is_dir() else "nothing"))
mounted = _found[0].parent
print("code dataset mounted at", mounted)

target = Path("/kaggle/working/btpvf-code")
_trees = [h for d in (0, 1, 2, 3)
          for h in sorted(mounted.glob("/".join(["*"] * d + ["pyproject.toml"])))]
if _trees:
    # /kaggle/input is read-only, so `pip install -e` cannot write its egg-info
    # there: the tree is copied into /kaggle/working first.
    shutil.copytree(_trees[0].parent, target, dirs_exist_ok=True)
    print("copied expanded snapshot from", _trees[0].parent)
elif (mounted / "btpvf_code.zip").is_file():
    with zipfile.ZipFile(mounted / "btpvf_code.zip") as zf:
        zf.extractall("/kaggle/working")
    print("unpacked btpvf_code.zip")
else:
    raise SystemExit("no code under %s: %s" % (
        mounted, sorted(p.name for p in mounted.iterdir())))

snapshot = json.loads((mounted / "btpvf_snapshot.json").read_text())
print("code snapshot:", json.dumps(snapshot, indent=2))
if not snapshot["citable"]:
    print("\n*** ITERATION MODE -- output of this run must NOT be cited in docs. ***\n")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(target) + "[resynth]"],
               check=True)

# An editable install is wired up by a .pth file, which only takes effect when an
# interpreter STARTS. This notebook's kernel started before the install, so the
# package is on disk but not importable until its source root is on sys.path.
sys.path.insert(0, str(target / "src"))
import importlib
importlib.invalidate_caches()
import experiments, data, vocoders  # noqa: F401  -- fail here, not three cells later
print("package importable from", experiments.__file__)


## Optional: a Hugging Face token

Anonymous downloads can be rate limited. Add a Kaggle Secret named `HF_TOKEN` and uncomment the two lines below. It changes nothing about *what* is downloaded, because every checkpoint is pinned by revision (INV-08). The package reads it from the environment only and never prints, logs or records it (INV-15), so it ships commented out.

In [ ]:
import os  # noqa: F401

# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Phase A, gated

Every gate runs: preflight (INV-08/13), alignment (INV-16), manifest validation, the analysis band limit (INV-17) and the sanity probes (INV-12). A failed gate writes no output directory; `--keep-failed` leaves staging aside as `<out>.failed` for diagnosis.

In [ ]:
import subprocess

from experiments.kaggle_harness import find_ljspeech_root

CONDITIONS = ['bigvgan_112m', 'vocos', 'griffin_lim']
CORPUS = 'auto'
OUT_DIR = '/kaggle/working/resynth_v0'
N_UTTERANCES = 200

# Kaggle mounts an attached dataset at a path that differs between
# images, so the corpus root is found by its shape unless one was given.
CORPUS = str(find_ljspeech_root()) if CORPUS == "auto" else CORPUS
print("corpus:", CORPUS)

subprocess.run(
    ["btpvf", "resynth", "--corpus-root", CORPUS, "--conditions", *CONDITIONS,
     "--out-dir", OUT_DIR, "--n-utterances", str(N_UTTERANCES), "--keep-failed"],
    check=True,
)

## Record which code produced this

The snapshot's mode, sha and content hash go into the manifest's provenance sidecar, so a dataset can never be read without knowing whether the code that made it is checkoutable.

In [ ]:
from experiments.kaggle_harness import record_code_provenance

# The mount is found by the file the snapshot always carries, not by a
# path: this account's kernels do not use the documented layout.
print(record_code_provenance(manifest=f"{OUT_DIR}/manifest.csv"))

## Reports and output

In [ ]:
subprocess.run(
    ["btpvf", "loudness", "--manifest", f"{OUT_DIR}/manifest.csv",
     "--out", "/kaggle/working/loudness_gain.csv"],
    check=True,
)
import shutil
shutil.make_archive(OUT_DIR, "zip", root_dir=OUT_DIR)